# Unwrapping AI — Zero to Transformer
## Step 04: Gradient Descent

**Creator:** Nitin K. Sethi  
**Learning format:** Concept → Implement → Visualise → Verify  
**Environment:** Google Colab | Python | NumPy | Matplotlib | scikit-learn

### Learning objectives
By the end of this notebook, you will be able to:
1. Explain why a model needs an optimisation algorithm.
2. Calculate gradients for a simple linear regression model.
3. Update model parameters using gradient descent.
4. Visualise how the loss changes during training.
5. Explain the effect of the learning rate.
6. Evaluate the trained model on unseen test data.

### The concept in one minute
A model starts with parameters that may produce poor predictions. Gradient descent calculates how the loss changes when each parameter changes, then adjusts the parameters in the direction that reduces the loss.

For a parameter vector **θ**:

**θ ← θ − learning_rate × gradient**

The learning rate controls the size of each update. Too small can make learning slow; too large can cause the loss to oscillate or diverge.

We will use a simple, reproducible synthetic regression dataset so that the optimisation process is easy to see. This notebook builds on the train–test split, MSE and linear regression concepts from Steps 01–03.


## 1. Import libraries and create a dataset

We create data with an underlying relationship close to:

**y = 7 + 4.5x + noise**

The model does not receive the true intercept or slope. It must estimate them from the training data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression

# Reproducibility
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

# Synthetic regression data
X = np.linspace(-3, 3, 200).reshape(-1, 1)
y = 7 + 4.5 * X.ravel() + rng.normal(0, 1.2, size=len(X))

# Keep test data unseen during training
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print(f"Training rows: {len(X_train)}")
print(f"Testing rows:  {len(X_test)}")
print("Feature shape:", X_train.shape)
print("Target shape: ", y_train.shape)

## 2. Visualise the training and testing data

Before training, inspect the data. The points should show an approximately linear relationship, with some noise.

In [ ]:
plt.figure(figsize=(9, 5))
plt.scatter(X_train.ravel(), y_train, alpha=0.7, label="Training data")
plt.scatter(X_test.ravel(), y_test, alpha=0.7, label="Testing data")
plt.xlabel("X")
plt.ylabel("y")
plt.title("Unwrapping AI — Synthetic Regression Dataset")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 3. Implement gradient descent from scratch

For simple linear regression:

- Prediction: **ŷ = θ₀ + θ₁x**
- Mean Squared Error: **MSE = (1/n) Σ(ŷ − y)²**
- Gradient for the parameters: **(2/n) Xᵀ(ŷ − y)**
- Update: **θ = θ − α × gradient**

Here, `θ₀` is the intercept, `θ₁` is the slope, and `α` is the learning rate.

We initialise both parameters to zero and update them repeatedly using only the training data.

In [ ]:
# Add a column of ones so the intercept is learned as a parameter
X_train_b = np.c_[np.ones((len(X_train), 1)), X_train]

# Initialise [intercept, slope]
theta = np.zeros(2)

learning_rate = 0.05
epochs = 1500
n = len(y_train)

loss_history = []

for epoch in range(epochs):
    predictions = X_train_b @ theta
    errors = predictions - y_train

    # Gradient of MSE with respect to intercept and slope
    gradients = (2 / n) * (X_train_b.T @ errors)

    # Parameter update
    theta = theta - learning_rate * gradients

    # Record loss after the update
    updated_predictions = X_train_b @ theta
    loss = np.mean((updated_predictions - y_train) ** 2)
    loss_history.append(loss)

print(f"Learned intercept: {theta[0]:.4f}")
print(f"Learned slope:     {theta[1]:.4f}")
print(f"Final training MSE: {loss_history[-1]:.4f}")

## 4. Observe the optimisation process

The loss curve should generally decrease and then flatten as the parameters approach a good solution. A flat curve means that further updates are making only small improvements.

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Training MSE")
plt.title("Unwrapping AI — Gradient Descent Learning Curve")
plt.grid(alpha=0.25)
plt.show()

print(f"Initial recorded MSE: {loss_history[0]:.4f}")
print(f"Final recorded MSE:   {loss_history[-1]:.4f}")

## 5. Plot the fitted regression line

The fitted line should follow the overall direction of the training observations. It will not pass through every point because the dataset includes noise.

In [ ]:
x_line = np.linspace(X_train.min(), X_train.max(), 100).reshape(-1, 1)
x_line_b = np.c_[np.ones((len(x_line), 1)), x_line]
y_line = x_line_b @ theta

plt.figure(figsize=(9, 5))
plt.scatter(X_train.ravel(), y_train, alpha=0.6, label="Training data")
plt.plot(x_line.ravel(), y_line, linewidth=2.5, label="Gradient descent fit")
plt.xlabel("X")
plt.ylabel("y")
plt.title("Unwrapping AI — Regression Line Learned by Gradient Descent")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 6. Evaluate on unseen test data

The test set was not used to calculate gradients or update parameters. We use it now to check how well the learned relationship generalises.

We also compare our result with scikit-learn's `LinearRegression`. The two solutions should be close because both minimise squared error for this one-feature linear regression problem.

In [ ]:
X_test_b = np.c_[np.ones((len(X_test), 1)), X_test]
y_pred_gd = X_test_b @ theta

gd_metrics = {
    "MSE": mean_squared_error(y_test, y_pred_gd),
    "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_gd)),
    "MAE": mean_absolute_error(y_test, y_pred_gd),
    "R2": r2_score(y_test, y_pred_gd)
}

sk_model = LinearRegression()
sk_model.fit(X_train, y_train)
y_pred_sk = sk_model.predict(X_test)

sk_metrics = {
    "MSE": mean_squared_error(y_test, y_pred_sk),
    "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_sk)),
    "MAE": mean_absolute_error(y_test, y_pred_sk),
    "R2": r2_score(y_test, y_pred_sk)
}

comparison = pd.DataFrame(
    [gd_metrics, sk_metrics],
    index=["Gradient Descent (from scratch)", "scikit-learn LinearRegression"]
)

display(comparison.round(4))

print("Gradient descent parameters:")
print(f"  Intercept = {theta[0]:.4f}")
print(f"  Slope     = {theta[1]:.4f}")
print("\nscikit-learn parameters:")
print(f"  Intercept = {sk_model.intercept_:.4f}")
print(f"  Slope     = {sk_model.coef_[0]:.4f}")

## 7. Experiment with the learning rate

The learning rate determines the size of each parameter update.

- **Small learning rate:** stable but may learn slowly.
- **Reasonable learning rate:** loss decreases efficiently.
- **Excessively large learning rate:** updates may overshoot and the loss may increase or diverge.

Run the experiment below. The intentionally large learning rate is included to demonstrate instability; it is not a recommended setting.

In [ ]:
def run_gradient_descent(rate, epochs=100):
    params = np.zeros(2)
    losses = []

    for _ in range(epochs):
        preds = X_train_b @ params
        errors = preds - y_train
        grads = (2 / n) * (X_train_b.T @ errors)
        params = params - rate * grads

        current_loss = np.mean((X_train_b @ params - y_train) ** 2)
        losses.append(current_loss)

        if not np.isfinite(current_loss) or current_loss > 1e8:
            break

    return params, losses

rates = [0.005, 0.05, 0.5]
plt.figure(figsize=(9, 5))

for rate in rates:
    _, losses = run_gradient_descent(rate)
    plt.plot(losses, label=f"Learning rate = {rate}")

plt.xlabel("Epoch")
plt.ylabel("Training MSE")
plt.title("Unwrapping AI — Comparing Learning Rates")
plt.yscale("log")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 8. Learning check and mini-challenge

Answer these in your own words:

1. What does the gradient tell the model?
2. Why do we subtract the gradient instead of adding it?
3. What can happen when the learning rate is too small?
4. What can happen when the learning rate is too large?
5. Why did we keep the test data out of the gradient updates?

### Mini-challenge
Change `learning_rate` to `0.01` and then `0.08`. Run the training cell and compare:
- the number of epochs needed to reach a low loss,
- the final training MSE,
- the test MSE.

Record which learning rate worked better for this dataset and explain why. Do not assume one learning rate is best for every dataset.

## 9. Verification checklist

Before marking Step 04 complete, verify all of the following:

- [ ] The notebook runs from the first cell to the last without errors.
- [ ] The training and testing row counts are 160 and 40.
- [ ] The learned intercept is reasonably close to 7.
- [ ] The learned slope is reasonably close to 4.5.
- [ ] The training loss decreases substantially.
- [ ] The fitted regression line follows the data trend.
- [ ] Gradient descent and scikit-learn produce similar test metrics.
- [ ] The learning-rate experiment shows why update size matters.
- [ ] I can explain the five learning-check questions.

### Step 04 takeaway
Gradient descent is an optimisation algorithm: it repeatedly adjusts model parameters using gradients to reduce a loss function. It is a central building block behind training neural networks and, ultimately, modern Transformer models.

**Next milestone:** Step 05 — Overfitting.

---
**Unwrapping AI — Learn the concept. Build the model. Understand the why.**